# ASSIGNMENT: Cartoon Face Mask
## This assignment tests your ability to apply face recognition, image masks, image thresholding, video capturing and saving video feed into a video file concepts.

## TASK: Cartoonify faces in video feed from live webcam

### Steps
- 1. **Capture video** feed from webcam
- 2. **Recognize faces** in the video
- 3. **Replace/Mask the face** region with your favorite cartoon character
- 4. **Save the video** feed into a video file
- 5. Submit the following files
    - Notebook (.ipynb)
    - Video (.avi/.mp4)

### Helper code to recognize faces

In [1]:
import cv2
import os
import urllib.request
import numpy as np

In [2]:
cartoon = cv2.imread("cartoon2.png", cv2.IMREAD_UNCHANGED)
cartoon.shape

(600, 600, 4)

In [3]:
url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_alt2.xml"
urllib.request.urlretrieve(url, "haarcascade_frontalface_alt2.xml")

cascPathface = "haarcascade_frontalface_alt2.xml"
faceCascade = cv2.CascadeClassifier(cascPathface)
faceCascade.empty()

False

In [4]:
video_capture = cv2.VideoCapture(0)

2026-07-24 12:02:43.945 python[8729:163219650] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.


In [5]:
ret, frame = video_capture.read()
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
faces = faceCascade.detectMultiScale(gray,
                                     scaleFactor=1.1,
                                     minNeighbors=5,
                                     minSize=(60, 60),
                                     flags=cv2.CASCADE_SCALE_IMAGE)
faces

()

In [6]:
def overlay_cartoon(frame, x, y, w, h, cartoon):
    ch, cw = cartoon.shape[:2]
    target_ratio = w / h
    current_ratio = cw / ch

    if current_ratio > target_ratio:
        new_cw = int(ch * target_ratio)
        start = (cw - new_cw) // 2
        cartoon_cropped = cartoon[:, start:start+new_cw]
    else:
        new_ch = int(cw / target_ratio)
        start = (ch - new_ch) // 2
        cartoon_cropped = cartoon[start:start+new_ch, :]

    cartoon_resized = cv2.resize(cartoon_cropped, (w, h))

    cartoon_bgr = cartoon_resized[:, :, :3]
    alpha_mask = cartoon_resized[:, :, 3]
    inv_mask = cv2.bitwise_not(alpha_mask)

    roi = frame[y:y+h, x:x+w]
    background = cv2.bitwise_and(roi, roi, mask=inv_mask)
    foreground = cv2.bitwise_and(cartoon_bgr, cartoon_bgr, mask=alpha_mask)

    frame[y:y+h, x:x+w] = cv2.add(background, foreground)
    return frame

In [7]:
cartoon = cv2.imread("cartoon2.png", cv2.IMREAD_UNCHANGED)
cartoon.shape

(600, 600, 4)

In [8]:
cartoon.shape[2]

4

In [9]:
cartoon[:,:,3].min(), cartoon[:,:,3].max()

(np.uint8(0), np.uint8(255))

In [10]:
for (x, y, w, h) in faces:
    frame = overlay_cartoon(frame, x, y, w, h, cartoon)

cv2.imshow("Cartoon Mask Test", frame)
for i in range(1, 20):
    cv2.waitKey(100)
cv2.destroyAllWindows()

In [11]:
while True:
    ret, frame = video_capture.read()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = faceCascade.detectMultiScale(gray,
                                         scaleFactor=1.1,
                                         minNeighbors=5,
                                         minSize=(60, 60),
                                         flags=cv2.CASCADE_SCALE_IMAGE)
    for (x, y, w, h) in faces:
        frame = overlay_cartoon(frame, x, y, w, h, cartoon)

    cv2.imshow('Cartoon Face Mask', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video_capture.release()
cv2.destroyAllWindows()

In [12]:
cv2.destroyAllWindows()
cv2.waitKey(1)

-1

In [13]:
def save_cartoon_video(outPath, fps):
    cap = cv2.VideoCapture(0)

    width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)

    fourcc = cv2.VideoWriter_fourcc(*"avc1")
    out = cv2.VideoWriter(outPath, fourcc, fps, (int(width), int(height)))

    while (cap.isOpened()):
        ret, frame = cap.read()

        if ret == True:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = faceCascade.detectMultiScale(gray,
                                                 scaleFactor=1.1,
                                                 minNeighbors=5,
                                                 minSize=(60, 60),
                                                 flags=cv2.CASCADE_SCALE_IMAGE)
            for (x, y, w, h) in faces:
                frame = overlay_cartoon(frame, x, y, w, h, cartoon)

            out.write(frame)
            cv2.imshow('Cartoon Face Mask', frame)
        else:
            break

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()

def main():
    save_cartoon_video('cartoon_output.mp4', 20.0)

if __name__ == '__main__':
    main()

In [14]:
import os
os.path.getsize("cartoon_output.mp4")

4263897